In [36]:
import kagglehub
path = kagglehub.dataset_download("jessicali9530/celeba-dataset")

Using Colab cache for faster access to the 'celeba-dataset' dataset.


In [37]:
import torchvision
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [38]:
class ImageProcessor:
  def __init__(self, root_path, transformation=None):
    self.root_path = root_path
    self.transformation = transformation

    self.all_img_paths = [os.path.join(root_path, img) for img in os.listdir(root_path)]

  def __len__(self):
    return len(self.all_img_paths)

  def __getitem__(self, idx):
    img_path = self.all_img_paths[idx]
    img = Image.open(img_path).convert("RGB")

    if self.transformation:
      img = self.transformation(img)

    return img

In [39]:
root_dir_path = os.path.join(path, "img_align_celeba/img_align_celeba")

In [40]:
transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178x218 =>178x178
    transforms.Resize(64), # 64x64
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # [-1,1]
])

In [41]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [42]:
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

## Generator Network

In [43]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [44]:
class Generator(nn.Module):
  def __init__(self, z_dim=100, img_channels = 3): # 3 is for RGB
    super(Generator, self).__init__()

    # Fully connected Layers
    self.model = nn.Sequential(
        nn.Linear(z_dim, 256),
        nn.ReLU(),

        nn.Linear(256, 512),
        nn.ReLU(),

        nn.Linear(512, 1024),
        nn.ReLU(),

        nn.Linear(1024, 64*64*img_channels),
        nn.Tanh() # [-1,1]
    )

  def forward(self,z):
    img = self.model(z)
    img = img.view(img.size(0), 3, 64, 64)
    return img # fake img => 64 x 64 x 3 x batch size (4D)

## Discriminator Network

In [45]:
class Discriminator(nn.Module):
  def __init__(self, img_channels = 3): # 3 is for RGB
    super(Discriminator, self).__init__()

    # Fully connected Layers
    self.model = nn.Sequential(
        nn.Flatten(), # 4D tensor => 1D

        nn.Linear(img_channels*64*64, 1024),
        nn.LeakyReLU(0.2, inplace=True),

        nn.Linear(1024, 512),
        nn.LeakyReLU(0.2, inplace=True),

        nn.Linear(512, 256),
        nn.LeakyReLU(0.2, inplace=True),

        nn.Linear(256, 1),
        nn.Sigmoid() # probablity for real or fake -> Binary classification
    )

  def forward(self, img):
    return self.model(img)

In [46]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5,0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5,0.999))

In [47]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [48]:
generator = generator.to(device)
discriminator = discriminator.to(device)

## Training the GANs

In [49]:
def train(generator, discriminator, dataloader, epochs=10):

  for epoch in range(epochs):
    for i, imgs in enumerate(dataloader):
      real_imgs = imgs.to(device)
      batch_size = real_imgs.size(0)

      # create labels for real and fake images
      real_labels = torch.ones(batch_size, 1).to(device)
      fake_labels = torch.zeros(batch_size, 1).to(device)

      # Train the Discriminator
      d_optimizer.zero_grad()

      fake_imgs = generator(torch.randn(batch_size, 100).to(device))

      real_loss = GAN_loss(discriminator(real_imgs), real_labels)
      fake_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

      d_loss = (real_loss + fake_loss) / 2

      d_loss.backward()
      d_optimizer.step()

      # Train the Generator
      g_optimizer.zero_grad()

      g_loss = GAN_loss(discriminator(fake_imgs), real_labels)

      g_loss.backward()
      g_optimizer.step()

      if i % 50 == 0:
        print(f"for epoch: {epoch+1}/{epochs}... batch: {i+1}... G-loss: {g_loss}... D_loss: {d_loss}")

    # save generated imgs for each epoch
    save_generated_images(generator, epoch, device)

In [50]:
import matplotlib.pyplot as plt
import torchvision
import numpy as np

def save_generated_images(generator, epoch, device, num_img = 8):
  z = torch.randn(num_img, 100).to(device)
  generated_imgs = generator(z).detach().cpu()

  grid = torchvision.utils.make_grid(generated_imgs, nrow=4, normalize=True)

  plt.imshow(np.transpose(grid, (1, 2, 0)))
  plt.title(f"epoch {epoch+1}")
  plt.axis("off")
  plt.savefig(f"epoch_{epoch+1}.png")
  plt.show()

In [ ]:
train(generator, discriminator, dataloader, 5)

for epoch: 1/5... batch: 1... G-loss: 0.7174142599105835... D_loss: 0.6847398281097412
for epoch: 1/5... batch: 51... G-loss: 0.9530065059661865... D_loss: 0.27564719319343567
